# Probability Theory and Probabilistic Models

In this lab, you will apply Bayesian reasoning and Monte Carlo simulation to behavioral assessment data. The primary exercise involves implementing Bayesian updating for functional assessment: given data from a functional analysis, you will quantify how evidence accumulates in favor of one behavioral function over others.

## Background

A standard functional analysis (FA; Iwata et al., 1982/1994) arranges four conditions -- attention, escape, tangible, and play (control) -- to identify the maintaining variable for problem behavior. Traditionally, clinicians rely on visual analysis to determine which condition produces elevated responding. A Bayesian approach offers a principled, quantitative alternative: we start with prior beliefs about the function (e.g., uniform across all four) and update those beliefs after each session based on the observed rates.

In the second part of the lab, you will use Monte Carlo simulation to estimate the sampling distribution and confidence intervals for a behavioral parameter, illustrating how simulation-based methods complement analytic approaches.

## Task 1: Import Libraries

Import the libraries you will need: `pandas`, `numpy`, `matplotlib.pyplot`, and `scipy.stats`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

np.random.seed(42)

## Task 2: Load and Inspect the FA Data

Load `functional_analysis_data.csv` into a DataFrame. Examine the structure of the data.

- How many sessions are there for each condition?
- What are the mean rates per minute for each condition?
- Based on visual inspection of the summary statistics alone, which function appears most likely?

In [ ]:
df = pd.read_csv("functional_analysis_data.csv")
print("Shape:", df.shape)
print(df.head())

print("\nSessions per condition:")
print(df['condition'].value_counts())

print("\nMean rate per minute by condition:")
print(df.groupby('condition')['rate_per_min'].agg(['mean', 'std', 'count']))

## Task 3: Traditional Visual Analysis

Create a standard FA graph: plot rate per minute (y-axis) as a function of session number (x-axis), with different markers and colors for each condition. Connect data points within each condition with lines.

This is the traditional approach to interpreting FA data. Note which condition shows consistently elevated responding.

In [ ]:
conditions = ['attention', 'escape', 'tangible', 'play']
markers = {'attention': 'o', 'escape': 's', 'tangible': '^', 'play': 'D'}

fig, ax = plt.subplots(figsize=(11, 6))
for cond in conditions:
    sub = df[df['condition'] == cond].sort_values('session')
    ax.plot(sub['session'], sub['rate_per_min'],
            marker=markers[cond], label=cond)
ax.set_xlabel("Session")
ax.set_ylabel("Rate per minute")
ax.set_title("Functional Analysis: Rate by Session and Condition")
ax.legend(title="Condition")
plt.tight_layout()
plt.show()

## Task 4: Set Up the Prior Distribution

Define a uniform prior probability distribution over four possible behavioral functions: `attention`, `escape`, `tangible`, and `automatic`.

Store this as a dictionary mapping each function name to its prior probability. Since we are starting with no prior information, each function should have equal probability.

Create a bar plot showing the prior distribution.

In [ ]:
functions = ['attention', 'escape', 'tangible', 'automatic']
prior = {f: 1.0 / len(functions) for f in functions}
print("Prior:", prior)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(prior.keys(), prior.values(), color='gray')
ax.set_ylim(0, 1)
ax.set_ylabel("P(function)")
ax.set_title("Prior Distribution (uniform)")
plt.tight_layout()
plt.show()

## Task 5: Define the Likelihood Function

To perform Bayesian updating, you need a likelihood function: $P(\text{data} | \text{function})$. Here is one reasonable approach:

For a given session, if the observed condition is the one matching the hypothesized function, the likelihood is proportional to the observed rate (higher rates = stronger evidence for that function). If the observed condition is the **play** (control) condition, a high rate provides weak evidence against all specific functions. If the observed condition does not match the hypothesized function, the likelihood should be lower.

Implement a likelihood function with the following logic:
- If the session condition matches the hypothesized function, likelihood = `rate + 0.1` (add a small constant to avoid zeros)
- If the session condition is play, likelihood = `1.0` (uninformative)
- If the session condition does not match and is not play, likelihood = `1.0 / (rate + 0.1)` (high rates in other conditions are weak evidence against)

**Note:** This is a simplified likelihood for pedagogical purposes. Research applications would use more principled statistical models.

In [ ]:
def likelihood(function, condition, rate):
    """P(data | function) for a single session under the simplified model."""
    if condition == function:
        return rate + 0.1
    elif condition == 'play':
        return 1.0
    else:
        return 1.0 / (rate + 0.1)

# Quick sanity check: an attention session with a high rate should most favor 'attention'
print({f: round(likelihood(f, 'attention', 8.4), 3) for f in functions})

## Task 6: Implement Bayesian Updating

Write a function that takes the current prior, a single session's data (condition and rate), and returns the updated posterior using Bayes' theorem:

$$P(\text{function} | \text{data}) = \frac{P(\text{data} | \text{function}) \cdot P(\text{function})}{P(\text{data})}$$

where $P(\text{data}) = \sum_{f} P(\text{data} | f) \cdot P(f)$ is the normalizing constant.

Then, loop through all sessions in chronological order, updating the posterior after each session. Store the posterior after each update so you can plot how beliefs evolve over time.

**Hint:** The posterior from session $n$ becomes the prior for session $n+1$.

In [ ]:
def bayesian_update(current, condition, rate):
    """Return the posterior dict after observing one session."""
    unnormalized = {f: likelihood(f, condition, rate) * current[f] for f in current}
    evidence = sum(unnormalized.values())
    return {f: unnormalized[f] / evidence for f in current}

# Loop through sessions in chronological order, carrying the posterior forward
posterior = dict(prior)
history = [dict(session=0, **posterior)]
for _, row in df.sort_values('session').iterrows():
    posterior = bayesian_update(posterior, row['condition'], row['rate_per_min'])
    history.append(dict(session=int(row['session']), **posterior))

history_df = pd.DataFrame(history)
print(history_df.tail())

## Task 7: Plot the Evolution of Posterior Beliefs

Create a figure showing how the posterior probability for each function changes across sessions. The x-axis should be session number, and the y-axis should be posterior probability (0 to 1). Plot a separate line for each function.

At what point does the Bayesian analysis converge on the correct function? How does this compare to how many sessions a visual analyst might need?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for f in functions:
    ax.plot(history_df['session'], history_df[f], marker='o', markersize=3, label=f)
ax.set_xlabel("Session")
ax.set_ylabel("Posterior probability")
ax.set_ylim(0, 1)
ax.set_title("Evolution of Posterior Beliefs Across Sessions")
ax.legend(title="Function")
plt.tight_layout()
plt.show()

## Task 8: Final Posterior and Comparison

Display the final posterior distribution as a bar plot. Place it side-by-side with the prior distribution so the shift in beliefs is clear.

Report the final posterior probabilities for each function. What is the Bayes factor comparing the most likely function to the next most likely? (The Bayes factor is the ratio of their posterior odds.)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ax1.bar(prior.keys(), prior.values(), color='gray')
ax1.set_title("Prior"); ax1.set_ylabel("Probability"); ax1.set_ylim(0, 1)
ax2.bar(posterior.keys(), posterior.values(), color='steelblue')
ax2.set_title("Final Posterior")
plt.tight_layout()
plt.show()

print("Final posterior:")
for f, p in sorted(posterior.items(), key=lambda kv: -kv[1]):
    print(f"  {f}: {p:.4f}")

ranked = sorted(posterior.values(), reverse=True)
bayes_factor = ranked[0] / ranked[1]
print(f"\nBayes factor (top vs next most likely): {bayes_factor:.1f}")

## Task 9: Monte Carlo Simulation for Confidence Intervals

Now shift to a different application of probabilistic reasoning. Suppose you want to estimate the mean rate of problem behavior in the attention condition and construct a confidence interval, but you want to use simulation rather than analytic formulas.

Implement a bootstrap Monte Carlo procedure:

1. Extract the observed rates from the attention condition.
2. Set the number of bootstrap samples to 10,000.
3. For each bootstrap iteration, resample the attention-condition rates **with replacement** (same sample size as original) and compute the mean.
4. Store all 10,000 bootstrap means.
5. Compute the 95% confidence interval using the 2.5th and 97.5th percentiles of the bootstrap distribution.
6. Plot a histogram of the bootstrap means with vertical lines marking the confidence interval bounds and the observed sample mean.

In [ ]:
attention_rates = df.loc[df['condition'] == 'attention', 'rate_per_min'].values
n = len(attention_rates)
n_boot = 10000

boot_means = np.array([
    np.random.choice(attention_rates, size=n, replace=True).mean()
    for _ in range(n_boot)
])

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
obs_mean = attention_rates.mean()
print(f"Observed mean: {obs_mean:.2f}")
print(f"95% bootstrap CI: [{ci_low:.2f}, {ci_high:.2f}]")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(boot_means, bins=40, color='steelblue', alpha=0.7)
ax.axvline(obs_mean, color='black', lw=2, label=f"observed mean = {obs_mean:.2f}")
ax.axvline(ci_low, color='red', ls='--', label=f"2.5% = {ci_low:.2f}")
ax.axvline(ci_high, color='red', ls='--', label=f"97.5% = {ci_high:.2f}")
ax.set_xlabel("Bootstrap mean rate (attention)")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of the Attention-Condition Mean")
ax.legend()
plt.tight_layout()
plt.show()

## Task 10: Interpretation and Discussion

In a markdown cell below, address the following questions:

1. What are the advantages of Bayesian updating over traditional visual analysis for FA interpretation? What are the disadvantages or limitations?
2. How sensitive was the Bayesian analysis to your choice of likelihood function? What would happen if you used a different likelihood specification?
3. How does the bootstrap confidence interval for the attention-condition mean compare to what you would get from a parametric approach (e.g., assuming normality)? You may compute both and compare.
4. In what applied scenarios might a probabilistic approach to functional assessment be particularly valuable (e.g., when visual analysis is ambiguous, when data are limited)?